In [110]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as ply
%matplotlib inline

In [111]:
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [112]:
# Build mapping system
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [113]:
# Build the dataset
block_size = 3
X, Y = [], [] # Context and label
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
        print("".join(itos[i] for i in context), '--->', itos[ix])
X = torch.tensor(X)
Y = torch.tensor(Y)
print(f"\nX shape: {X.shape}")
print(f"Y shape: {Y.shape}")

emma
..e ---> e
.em ---> m
emm ---> m
mma ---> a
ma. ---> .
olivia
..o ---> o
.ol ---> l
oli ---> i
liv ---> v
ivi ---> i
via ---> a
ia. ---> .
ava
..a ---> a
.av ---> v
ava ---> a
va. ---> .
isabella
..i ---> i
.is ---> s
isa ---> a
sab ---> b
abe ---> e
bel ---> l
ell ---> l
lla ---> a
la. ---> .
sophia
..s ---> s
.so ---> o
sop ---> p
oph ---> h
phi ---> i
hia ---> a
ia. ---> .

X shape: torch.Size([32, 3])
Y shape: torch.Size([32])


Check Bengio et. Al, 2003 paper and Andej Karpathy's walkthrough to better clarify architecture. Key points:
- The model looks at 4 words, and aims to predict the 5th.
- Each word is given a 30 dimension vector embedding.
- Every (17000) word's embedding is stored inside a 17000 by 30 lookup matrix, C, which is used to map each of the 4 previous words to their embeddings, before being fed into the neural network to get predictions for the 5th word. 

For our experiment, we are going to make letter embeddings 2 dimensions, since there are only 27 possibilities compared to 17000 words in the paper.

In [114]:
C = torch.randn((27, 2)) # Lookup matrix contains a 2 dimension embedding for each of the 27 characters

In [115]:
C[X] # Lookup the embedding for each character in X
# The output is the embedding (2d) for each character in X (32 inputs x 3 characters x 2 dimensions)

tensor([[[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.3148,  0.4877]],

        [[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.6250, -1.0044]],

        [[-0.3148,  0.4877],
         [-0.6250, -1.0044],
         [-0.2372,  2.0049]],

        [[-0.6250, -1.0044],
         [-0.2372,  2.0049],
         [-0.2372,  2.0049]],

        [[-0.2372,  2.0049],
         [-0.2372,  2.0049],
         [ 0.0620, -0.1603]],

        [[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.3148,  0.4877]],

        [[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.4404,  0.2556]],

        [[-0.3148,  0.4877],
         [-0.4404,  0.2556],
         [ 0.5240,  0.5988]],

        [[-0.4404,  0.2556],
         [ 0.5240,  0.5988],
         [ 0.7131, -0.9109]],

        [[ 0.5240,  0.5988],
         [ 0.7131, -0.9109],
         [-1.3851, -2.3727]],

        [[ 0.7131, -0.9109],
         [-1.3851, -2.3727],
         [ 0.7131, -0.9109]],

        [[-1.3851, -2

In [116]:
C[X].shape

torch.Size([32, 3, 2])

In [117]:
emb = C[X]

Hidden layer

In [118]:
W1 = torch.randn((6, 100)) # 3x2 weights per neuron, 100 neurons
b1 = torch.randn(100) # 100 bias terms

In [119]:
# What we'd like to do:
# emb @ W1 + b1
# This won't work since emb is 32x3x2 and W1 is 6x100
# We must reshape emb from 32x3x2 into 32x6
# We can use the .view() method to reshape the tensor

In [120]:
W1.shape

torch.Size([6, 100])

In [121]:
h = emb.view((-1), 6) @ W1 + b1

OR

In [122]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

In [123]:
# torch.cat(torch.unbind(emb, 1), 1)

Between emb.view and torch.cat, we generally opt for emb.view, since its much more storage and compute efficient.

In [124]:
h = torch.tanh(h)
h

tensor([[ 0.4693, -0.9810,  0.8856,  ..., -0.2102, -0.8302, -0.9674],
        [-0.9172, -0.9342,  0.6112,  ..., -0.0096,  0.9880, -0.8842],
        [ 0.9923, -0.9989,  0.9640,  ...,  0.8109, -0.9999, -0.9942],
        ...,
        [ 0.9795,  0.7458,  0.8597,  ...,  0.9167, -0.9366, -0.4570],
        [ 0.9363,  0.9998, -0.3937,  ..., -0.6310,  0.9527,  0.9996],
        [ 0.9808,  0.9694, -0.4742,  ..., -0.2520,  0.3384,  0.9997]])

Output layer

In [126]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [127]:
logits = h @ W2 + b2

In [130]:
# Treat the logits as log values that need to be exponentiated
counts = logits.exp()

In [131]:
probs = counts / counts.sum(1, keepdims=True)

In [137]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(17.1183)

In [135]:
probs

tensor([[3.6797e-03, 4.4458e-13, 3.1485e-08, 6.3839e-10, 1.7673e-09, 1.0683e-10,
         1.7177e-06, 3.0705e-12, 2.6325e-06, 8.2961e-07, 9.1174e-09, 9.0627e-10,
         9.9607e-15, 2.6461e-16, 5.5450e-09, 1.7974e-08, 1.1973e-07, 1.2667e-15,
         2.4024e-08, 9.9630e-01, 4.9516e-12, 9.5413e-08, 1.6751e-05, 1.3463e-09,
         6.2880e-11, 1.3244e-08, 6.6357e-11],
        [9.4713e-03, 8.4915e-11, 1.2639e-05, 5.6424e-10, 1.2622e-07, 1.2139e-05,
         4.1178e-04, 1.2407e-13, 5.2644e-08, 1.5410e-06, 6.0161e-09, 1.0184e-08,
         4.2715e-06, 2.3555e-12, 5.9427e-04, 2.5032e-03, 1.2983e-02, 8.0718e-09,
         1.1744e-04, 9.7305e-01, 6.8645e-11, 5.8320e-06, 3.5127e-06, 7.1111e-04,
         2.7582e-05, 8.9041e-05, 4.6976e-09],
        [2.8899e-07, 1.2956e-18, 1.2117e-14, 1.0640e-11, 1.2094e-15, 7.8732e-20,
         4.6680e-16, 1.4794e-18, 1.8326e-10, 8.8675e-16, 9.9584e-16, 7.1873e-17,
         5.9743e-22, 8.5796e-20, 4.9915e-17, 8.0034e-18, 9.3164e-16, 9.6904e-23,
         8.4103e-

All together:

In [138]:
# Generating random weights for the entire network
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [140]:
sum(p.nelement() for p in parameters)

3481

In [141]:
# Forward pass
emb = C[X]
h = torch.tanh(emb.view((-1), 6) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)

In [142]:
F.cross_entropy(logits, Y)

tensor(17.7697)

In [145]:
for p in parameters:
    p.requires_grad = True

# training loop
for _ in range(100):

    # Forward pass
    emb = C[X]
    h = torch.tanh(emb.view((-1), 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    # Bakward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update the parameters
    with torch.no_grad():
        for p in parameters:
            p -= 0.1 * p.grad

3.985849142074585
3.602830410003662
3.2621419429779053
2.961381196975708
2.6982970237731934
2.469712734222412
2.271660804748535
2.1012840270996094
1.957176923751831
1.8374861478805542
1.7380965948104858
1.653511881828308
1.5790901184082031
1.511767029762268
1.4496052265167236
1.3913124799728394
1.3359930515289307
1.2830536365509033
1.232191801071167
1.1833821535110474
1.1367992162704468
1.0926648378372192
1.0510929822921753
1.0120275020599365
0.9752705693244934
0.9405568242073059
0.9076130986213684
0.8761922121047974
0.8460891246795654
0.8171360492706299
0.78919917345047
0.7621749043464661
0.7359816431999207
0.7105581760406494
0.6858612298965454
0.6618653535842896
0.6385658383369446
0.6159819960594177
0.594166100025177
0.5732106566429138
0.5532564520835876
0.5344885587692261
0.5171172022819519
0.501331627368927
0.48724299669265747
0.4748406410217285
0.4639979302883148
0.45451465249061584
0.4461711645126343
0.43876639008522034
0.4321332573890686
0.4261389970779419
0.4206800162792206
0.4

In [146]:
logits

tensor([[ 5.4521e+00,  1.1406e+01,  6.0999e+00,  3.8516e+00, -1.9990e+00,
          1.1359e+01, -1.8854e+01,  5.1638e+00, -3.9998e+00,  1.1395e+01,
          6.6218e+00, -1.3545e+00,  5.8592e-01,  7.8595e+00,  1.5338e+00,
          1.1317e+01,  1.9508e+00,  7.0253e+00, -7.6385e-01,  1.1384e+01,
         -5.2119e-01,  5.7447e+00,  4.0099e+00, -6.0182e+00,  7.7466e+00,
         -4.1840e+00,  6.1011e+00],
        [-4.9741e+00,  7.2376e+00,  2.7345e+00,  2.2702e+00,  6.9653e+00,
          8.8068e+00, -3.9786e+00,  5.1359e+00,  7.7141e+00, -3.2167e+00,
          9.4259e+00,  6.7037e-01,  1.0925e+01,  1.3284e+01,  4.5206e+00,
          5.4684e+00,  7.7529e+00,  3.6895e+00,  6.8494e+00,  8.8477e+00,
         -8.3920e-03, -7.1254e+00, -1.1025e+01, -8.6388e+00,  1.5988e+00,
         -2.7511e+00,  7.1019e+00],
        [ 7.1084e+00,  1.0056e+01,  1.2959e+01,  4.4468e+00, -3.1591e+00,
          5.5343e+00, -9.5849e+00, -8.5539e+00, -1.1084e+01,  1.4484e+01,
          5.7549e+00,  1.1865e+01,  4.37

In [147]:
logits.max(1)

torch.return_types.max(
values=tensor([11.4056, 13.2839, 19.0095, 17.7593, 12.9712, 11.4056, 13.1443, 11.7389,
        13.6228, 15.5924, 12.7179, 17.7828, 11.4056, 12.9591, 14.1629, 17.1673,
        11.4056, 13.9451, 11.5875, 13.3441, 15.9006, 12.3487,  8.0381,  8.0298,
        13.9526, 11.4056, 13.3292, 13.7648, 11.3003, 14.3217, 15.7601, 12.2277],
       grad_fn=<MaxBackward0>),
indices=tensor([ 1, 13, 13,  1,  0,  1, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  1, 19,
         1,  2,  5, 12, 12,  1,  0,  1, 15, 16,  8,  9,  1,  0]))

In [148]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

We can compare the highest probability output from the logits and it's respective index. For example, the first (0th) value of logits.max(1) was 11.4056, which corresponds to index 1. Looking at the first index of Y (the ground truth), the first index is 5. In a perfectly fitted model (with a loss of 0), the indices of each of the max logits should be identical to the ground truth indices. Notive how the mox logits for indices 1 and 2 are both 13, just like the ground truth set. A loss of 0 is impossible though, since looking at the visualized breakdown of the first few words (higher up in this notebook), each name's first input block is '...', and the output is a specific letter. The some input blocks can be identical for different names, but lead to different outputs (next letters). For example with the names Ava and Isabella. The first input block for both (all names actually) is '...'. However, for Ava, the predicted next letter is A, and for Isabella, the predicted next letter is I. Notice how an input of '...' can lead to multiple possible outcomes?

Now that we understand, lets optimize the neural network for the entire dataset of 32 000 names.

In [159]:
# Build the dataset
block_size = 3
X, Y = [], [] # Context and label
for w in words:
    # print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
        # print("".join(itos[i] for i in context), '--->', itos[ix])
X = torch.tensor(X)
Y = torch.tensor(Y)
# print(f"\nX shape: {X.shape}")
# print(f"Y shape: {Y.shape}")
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [160]:
# Reinitialize weights
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

# Find # of parameters
sum(p.nelement() for p in parameters)

3481

In [161]:
for p in parameters:
    p.requires_grad = True

# training loop
for _ in range(10):

    # Forward pass
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view((-1), 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    # Bakward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update the parameters with gradient descent
    with torch.no_grad():
        for p in parameters:
            p -= 0.1 * p.grad

19.505229949951172
17.084487915039062
15.776531219482422
14.833338737487793
14.002596855163574
13.253255844116211
12.579915046691895
11.983098983764648
11.470491409301758
11.05185604095459


Computing this, while pretty quick is actually slow considering all of the computation the computer has to do. Instead, lets make minibatches that are smaller, and quicker to train

In [163]:
for p in parameters:
    p.requires_grad = True

# Make minibatches
ix = torch.randint(0, X.shape[0], (32,))

# training loop
for _ in range(100):

    # Forward pass
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view((-1), 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    # Bakward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update the parameters with gradient descent
    with torch.no_grad():
        for p in parameters:
            p -= 0.1 * p.grad

9.5419282913208
8.375968933105469
7.401012420654297
6.571034908294678
5.852638244628906
5.235065937042236
4.699171543121338
4.225724697113037
3.8031091690063477
3.432532548904419
3.124642848968506
2.868403434753418
2.644404411315918
2.4430716037750244
2.258000373840332
2.085092544555664
1.9238882064819336
1.7757019996643066
1.639176845550537
1.511244773864746
1.3904170989990234
1.2767232656478882
1.1709531545639038
1.0742369890213013
0.9873625040054321
0.910121738910675
0.8417296409606934
0.7816551327705383
0.729709804058075
0.6855982542037964
0.6485703587532043
0.6174889802932739
0.5911523699760437
0.5685332417488098
0.5488445162773132
0.531503438949585
0.5160778164863586
0.5022403597831726
0.4897368550300598
0.4783657491207123
0.4679650068283081
0.45840221643447876
0.4495681822299957
0.4413720369338989
0.4337378442287445
0.4266013503074646
0.4199081063270569
0.4136119782924652
0.4076724350452423
0.40205562114715576
0.39673110842704773
0.3916732370853424
0.3868592381477356
0.382268846

In [164]:
h = torch.tanh(emb.view((-1), 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y[ix])
loss

tensor(0.2785, grad_fn=<NllLossBackward0>)